# Steam Recommender System - Master Pipeline

This notebook follows `project.md` end to end:

1. Task 1: data preparation and exploration
2. Task 2: Random and Popularity baselines
3. Task 3: MF-BPR and Two-Tower retrieval
4. Task 4: LightGBM ranking on retrieval candidates

The notebook uses the local modules in `src/` and `scripts/` so each stage stays separated but can still be run in one place.

## Colab setup

Run this block first in Google Colab. It mounts Drive for checkpoints, clones the repo, and sets a single `ROOT_PATH` used everywhere else in the notebook.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Replace with your repo URL if running in Colab.
REPO_URL = "https://github.com/<your-org-or-user>/steam-recsys-pipeline.git"
BRANCH = "main"
PROJECT_NAME = "steam-recsys-pipeline"
ROOT_PATH = f"/content/{PROJECT_NAME}"
CHECKPOINT_ROOT = Path('/content/drive/MyDrive') / PROJECT_NAME / 'checkpoints'
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

!rm -rf $ROOT_PATH
!git clone --branch $BRANCH $REPO_URL $ROOT_PATH

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path(ROOT_PATH) if Path(ROOT_PATH).exists() else Path('.').resolve()
SRC_ROOT = REPO_ROOT / 'src'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print('repo root:', REPO_ROOT)
print('src root:', SRC_ROOT)
print('checkpoint root:', CHECKPOINT_ROOT)

## Shared imports and paths

The notebook prefers the already processed splits. If you have the Task 1 raw Steam files, you can still rerun the prep module separately, but the master notebook works off the processed artifacts.

In [ ]:
import json
import time
from dataclasses import asdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from steam_recsys.data_prep import summarize
from steam_recsys.ranking.features import build_features
from steam_recsys.ranking.train import evaluate_ranker, prepare_ranking_data, save_model, train_ranker
from scripts.metrics import compute_all_metrics
from scripts.mf_bpr import MFBPRConfig, MFBPRRecommender
from scripts.models import PopularityRecommender, RandomRecommender
from scripts.plot_task3_results import load_results, plot_bars, plot_latency, plot_tradeoff
from scripts.evaluate_task3 import evaluate as evaluate_task3
from scripts.two_tower import TwoTowerConfig, TwoTowerRecommender

sns.set_theme(style='darkgrid')
plt.rcParams['figure.dpi'] = 110

DATA_DIRS = [REPO_ROOT / 'data/processed/steam', REPO_ROOT / 'processed_data', REPO_ROOT]
for candidate in DATA_DIRS:
    if (candidate / 'train.parquet').exists():
        DATA_DIR = candidate
        break
else:
    raise FileNotFoundError('Could not locate train.parquet in any expected data directory')

OUTPUT_DIR = REPO_ROOT / 'outputs' / 'master_pipeline'
PLOTS_DIR = OUTPUT_DIR / 'plots'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = CHECKPOINT_ROOT / 'models'
ARTIFACT_DIR = CHECKPOINT_ROOT / 'artifacts'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_DIR =', DATA_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('MODEL_DIR =', MODEL_DIR)
print('ARTIFACT_DIR =', ARTIFACT_DIR)

## Task 1 - Data preparation and exploration

Load the processed splits, inspect the schema, and summarize the dataset. This section documents the handoff from the preparation stage into the model stages.

In [ ]:
train = pd.read_parquet(DATA_DIR / 'train.parquet')
validation = pd.read_parquet(DATA_DIR / 'validation.parquet')
test = pd.read_parquet(DATA_DIR / 'test.parquet')
items_path = DATA_DIR / 'items.parquet'
catalog = pd.read_parquet(items_path) if items_path.exists() else pd.DataFrame({'item_id': sorted(train['item_id'].astype(str).unique())})

summary = summarize(train, catalog)
display(train.head())
print(json.dumps(summary, default=str, indent=2))
print('validation rows:', len(validation))
print('test rows:', len(test))

## Task 2 - Random and Popularity baselines

The baselines establish the lower bound and the strongest non-personalized reference.

In [ ]:
def build_history(*dfs: pd.DataFrame) -> dict[str, set[str]]:
    combined = pd.concat(dfs, ignore_index=True)
    return combined.groupby('user_id')['item_id'].apply(lambda s: set(s.astype(str))).to_dict()


def build_ground_truth(df: pd.DataFrame) -> dict[str, set[str]]:
    positives = df[df['is_positive']]
    return positives.groupby('user_id')['item_id'].apply(lambda s: set(s.astype(str))).to_dict()


val_history = build_history(train)
test_history = build_history(train, validation)
val_gt = build_ground_truth(validation)
test_gt = build_ground_truth(test)
catalog_set = set(train['item_id'].astype(str).unique())
val_users = [u for u, pos in val_gt.items() if pos]
test_users = [u for u, pos in test_gt.items() if pos]

random_model = RandomRecommender(seed=42).fit(train, item_col='item_id')
pop_model = PopularityRecommender().fit(train, item_col='item_id', label_col='is_positive', use_positive_only=True)

task2_rows = []
for split_name, users, history, gt in [('validation', val_users, val_history, val_gt), ('test', test_users, test_history, test_gt)]:
    for label, model in [('Random', random_model), ('Popularity', pop_model)]:
        t0 = time.perf_counter()
        recs = model.recommend(users, history, k=20)
        latency = time.perf_counter() - t0
        metrics = compute_all_metrics(recs, gt, catalog_set, recall_k=20, ndcg_k=10)
        task2_rows.append({'split': split_name, 'model': label, 'latency_sec': latency, **metrics})

task2_df = pd.DataFrame(task2_rows)
display(task2_df)

## Task 3 - MF-BPR and Two-Tower retrieval

This stage compares the collaborative retrievers required by the project specification.

In [ ]:
mf_config = MFBPRConfig(n_factors=64, learning_rate=0.05, reg=0.0025, epochs=5, batch_size=2048, seed=42, device='cpu')
tt_config = TwoTowerConfig(user_dim=64, item_dim=64, hidden_dim=128, batch_size=1024, epochs=5, learning_rate=1e-3, temperature=0.07, seed=42, device='cpu')

task3_results = evaluate_task3(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR / 'task3',
    k=20,
    ndcg_k=10,
    mf_config=mf_config,
    two_tower_config=tt_config,
    max_train_positives=200000,
)

task3_df = load_results(OUTPUT_DIR / 'task3' / 'task3_results.json')
display(task3_df)
plot_bars(task3_df, 'recall@20', OUTPUT_DIR / 'task3' / 'plots' / 'recall_at_20.png')
plot_bars(task3_df, 'ndcg@10', OUTPUT_DIR / 'task3' / 'plots' / 'ndcg_at_10.png')
plot_bars(task3_df, 'coverage@20', OUTPUT_DIR / 'task3' / 'plots' / 'coverage_at_20.png')
plot_latency(task3_df, OUTPUT_DIR / 'task3' / 'plots' / 'latency.png')
plot_tradeoff(task3_df, OUTPUT_DIR / 'task3' / 'plots' / 'coverage_vs_recall.png')

## Task 4 - LightGBM ranking on retrieval candidates

The ranking stage takes retrieval candidates, engineers features from training data only, and learns a LambdaRank model.
Here we generate candidates from the two-tower retriever trained above, with a FAISS top-k search when available and a numpy fallback otherwise.

In [ ]:
def build_retrieval_candidates(interactions: pd.DataFrame, history: dict[str, set[str]], users: list[str], retriever, k: int = 100) -> pd.DataFrame:
    rows = []
    recs = retriever.recommend(users, history, k=k)
    gt = build_ground_truth(interactions)
    user_events = interactions.groupby('user_id')['event_time'].max().to_dict()
    user_hours = interactions.groupby(['user_id', 'item_id'])['hours'].first().to_dict()
    item_avg_hours = train.groupby('item_id')['hours'].mean().to_dict()
    for user_id, items in recs.items():
        positives = gt.get(user_id, set())
        event_time = user_events.get(user_id, interactions['event_time'].max())
        for item_id in items:
            key = (user_id, item_id)
            hours = user_hours.get(key, item_avg_hours.get(item_id, 0.0))
            rows.append({
                'user_id': user_id,
                'item_id': str(item_id),
                'hours': float(hours),
                'event_time': event_time,
                'is_positive': str(item_id) in positives,
                'text_length': 0,
                'early_access_review': False,
                'products_owned': np.nan,
                'found_funny': 0,
                'received_for_free': False,
            })
    return pd.DataFrame(rows)


two_tower_model = TwoTowerRecommender(config=tt_config).fit(train, catalog, user_col='user_id', item_col='item_id', label_col='is_positive')

candidates_val = build_retrieval_candidates(validation, val_history, val_users, two_tower_model, k=100)
candidates_test = build_retrieval_candidates(test, test_history, test_users, two_tower_model, k=100)
df_val, feature_cols, encoder = build_features(candidates_val, train, catalog)
df_test, _, _ = build_features(candidates_test, train, catalog, encoder=encoder)

val_users_rank = df_val['user_id'].drop_duplicates().tolist()
split_user_idx = int(len(val_users_rank) * 0.8)
train_users_rank = set(val_users_rank[:split_user_idx])
valid_users_rank = set(val_users_rank[split_user_idx:])
df_rank_train = df_val[df_val['user_id'].isin(train_users_rank)].copy()
df_rank_valid = df_val[df_val['user_id'].isin(valid_users_rank)].copy()
X_train, y_train, q_train = prepare_ranking_data(df_rank_train, feature_cols)
X_valid, y_valid, q_valid = prepare_ranking_data(df_rank_valid, feature_cols)
X_test, y_test, q_test = prepare_ranking_data(df_test, feature_cols)

ranker = train_ranker(X_train, y_train, q_train, X_valid, y_valid, q_valid, feature_cols, num_boost_round=300, early_stopping_rounds=20, verbose_eval=50)
ranking_metrics = evaluate_ranker(ranker, X_test, y_test, q_test, item_ids=df_test['item_id'].values, catalog_size=len(catalog), k=10)
print(json.dumps(ranking_metrics, indent=2))
save_model(ranker, feature_cols, encoder=encoder, model_dir=str(OUTPUT_DIR / 'task4' / 'models'), artifact_dir=str(OUTPUT_DIR / 'task4' / 'artifacts'))

## Final summary

The notebook produces the data summary, baseline metrics, retrieval comparison, retrieval plots, and a ranking-stage experiment. You can copy the resulting tables and charts into `ANALYSIS.md` and the final report.